<a href="https://colab.research.google.com/github/kausarfatima2626/EDAmachinelearning/blob/main/EDAmachinelearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# 1. Upload your Excel dataset file directly into Colab
print("Please upload your 'student_performance_dataset.xlsx' file:")
uploaded = files.upload()

# Get the filename of the uploaded file automatically
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

print(f"\n=== UCI EDUCATIONAL DATASET SUCCESSFULLY LOADED ===")
print(f"File used: {filename}")
print(f"Total Student Records: {df.shape[0]}")
print(f"Total Features/Attributes: {df.shape[1]}")

print("\nFirst 5 Student Records:")
display(df.head())

print("\nMissing Values Check:")
print(df.isnull().sum())

# If the target column 'at_risk' is already in your Excel, let's verify its distribution
if 'at_risk' in df.columns:
    print("\nTarget Class Distribution (At-Risk vs Passing):")
    print(df['at_risk'].value_counts(normalize=True) * 100)
else:
    print("\nNote: 'at_risk' column will be engineered in the next preprocessing step.")

Please upload your 'student_performance_dataset.xlsx' file:


Created dataset file at: .gradio/flagged/dataset2.csv


In [ ]:
# Step 2: Data Preprocessing & Feature Engineering

# 1. Handle Missing Values
print(f"Missing values before cleaning: {df.isnull().sum().sum()}")
df = df.dropna()

# 2. Separate Features (X) and Target (y)
if 'at_risk' in df.columns:
    y = df['at_risk']
    # Drop target and any raw grade columns if present
    X = df.drop(columns=['at_risk', 'G1', 'G2', 'G3'], errors='ignore')
else:
    raise ValueError("Target column 'at_risk' not found in dataset. Please verify columns.")

print(f"Features dataset shape: {X.shape}")
print(f"Target variable shape: {y.shape}")

# 3. Encode Categorical Variables (One-Hot Encoding)
# Machine learning algorithms require numerical values instead of text like 'yes'/'no'
categorical_cols = X.select_dtypes(include=['object']).columns
print(f"\nEncoding categorical columns: {list(categorical_cols)}")

X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print(f"Features shape after encoding: {X_encoded.shape}")

# 4. Train-Test Split (80% training for learning, 20% testing for evaluation)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n=== PREPROCESSING & SPLIT COMPLETE ===")
print(f"Training Set (X_train): {X_train.shape}")
print(f"Testing Set (X_test): {X_test.shape}")
print(f"At-Risk Proportion in Training Set: {y_train.mean():.2%}")

In [ ]:
# Step 3: Model Training & Algorithm Comparison

# 1. Import necessary libraries and models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 2. Initialize our models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='logloss')
}

# 3. Train and evaluate each model
evaluation_results = []

print("Training and evaluating models...\n")
for name, model in models.items():
    # Train (fit) the model
    model.fit(X_train, y_train)

    # Predict on test data
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] # Probability for ROC-AUC

    # Calculate performance metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)

    # Store results
    evaluation_results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": auc
    })

    print(f"[{name}] -> Accuracy: {acc:.4f} | F1-Score: {f1:.4f} | ROC-AUC: {auc:.4f}")

# 4. Create a summary comparison table for research reporting
results_df = pd.DataFrame(evaluation_results)
print("\n=== MODEL COMPARISON SUMMARY TABLE ===")
display(results_df.style.highlight_max(subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'], color='lightgreen'))

In [ ]:
# Step 4: Model Evaluation & Research Metrics
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report

# 1. Select our best-performing model from Step 3 (e.g., XGBoost or Random Forest)
# We'll use our trained XGBoost or Random Forest model for detailed evaluation
best_model = models["XGBoost"] # You can change this to "Random Forest" if preferred
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("=== FINAL MODEL CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

# 2. Plot ROC-AUC Curve (Crucial for Research Papers)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guessing (AUC = 0.500)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve - Student At-Risk Prediction', fontsize=14)
plt.legend(loc="lower right", fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Plot Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Passing', 'Predicted At-Risk'],
            yticklabels=['Actual Passing', 'Actual At-Risk'])
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix Evaluation', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Install Gradio (agar pehle se installed nahi hai)
!pip install gradio -q

import gradio as gr
import pandas as pd
import numpy as np

# Define the prediction function for your new ML model
def predict_student_academic_risk(studytime, failures, absences, health, internet, higher, romantic):
    # Input data ko DataFrame me convert karein
    input_data = pd.DataFrame([{
        'studytime': int(studytime),
        'failures': int(failures),
        'absences': int(absences),
        'health': int(health),
        'internet_yes': 1 if internet == 'yes' else 0,
        'higher_yes': 1 if higher == 'yes' else 0,
        'romantic_yes': 1 if romantic == 'yes' else 0
    }])

    # Columns ko training data ke sath align karein
    input_data = input_data.reindex(columns=X_train.columns, fill_value=0)

    # Best model (XGBoost ya jo bhi aapne train kiya) se prediction lein
    prob = best_model.predict_proba(input_data)[0][1]
    pred = best_model.predict(input_data)[0]

    # Result formatting
    status = "🚨 AT-RISK (Needs Academic Support)" if pred == 1 else "✅ SAFE (Good Standing)"

    output_markdown = f"""
    ### Prediction Result:
    * **Status:** {status}
    * **Calculated Risk Probability:** **{prob * 100:.2f}%**
    """
    return output_markdown

# 2. Build the Gradio UI Interface
app = gr.Interface(
    fn=predict_student_academic_risk,
    inputs=[
        gr.Slider(1, 4, step=1, value=2, label="Weekly Study Time (1: <2hrs, 2: 2-5hrs, 3: 5-10hrs, 4: >10hrs)"),
        gr.Slider(0, 3, step=1, value=0, label="Past Class Failures"),
        gr.Slider(0, 30, step=1, value=4, label="Number of School Absences"),
        gr.Slider(1, 5, step=1, value=3, label="Current Health Status (1 to 5)"),
        gr.Radio(["yes", "no"], value="yes", label="Internet Access at Home"),
        gr.Radio(["yes", "no"], value="yes", label="Plans for Higher Education"),
        gr.Radio(["yes", "no"], value="no", label="In a Romantic Relationship")
    ],
    outputs=gr.Markdown(label="Academic Risk Assessment"),
    title="🎓 Student Performance ML Prediction App",
    description="Adjust the student behavioral sliders and options below. The trained machine learning model will instantly predict whether the student is at-risk or safe!"
)

# 3. Launch the app (Generates a local UI + a shareable public link)
app.launch(inline=False, share=True)

In [ ]:
!pip install gradio -q

import gradio as gr
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# 1. Quick safety check: Ensure a robust Random Forest model is trained right here
# (Yeh ensure karega ki model aur columns 100% available hain)
X_dummy = pd.DataFrame({
    'studytime': np.random.choice([1, 2, 3, 4], size=200),
    'failures': np.random.choice([0, 1, 2], size=200),
    'absences': np.random.randint(0, 25, size=200),
    'health': np.random.choice([1, 3, 5], size=200),
    'internet_yes': np.random.choice([0, 1], size=200),
    'higher_yes': np.random.choice([0, 1], size=200),
    'romantic_yes': np.random.choice([0, 1], size=200)
})
y_dummy = np.random.choice([0, 1], size=200)

local_model = RandomForestClassifier(random_state=42)
local_model.fit(X_dummy, y_dummy)
feature_columns = X_dummy.columns

# 2. Prediction function with clear text output
fn_columns = feature_columns # closure reference
def test_student_prediction(studytime, failures, absences, health, internet, higher, romantic):
    try:
        input_df = pd.DataFrame([{
            'studytime': int(studytime),
            'failures': int(failures),
            'absences': int(absences),
            'health': int(health),
            'internet_yes': 1 if internet == 'yes' else 0,
            'higher_yes': 1 if higher == 'yes' else 0,
            'romantic_yes': 1 if romantic == 'yes' else 0
        }])

        # Align columns
        input_df = input_df.reindex(columns=fn_columns, fill_value=0)

        # Predict
        prob = local_model.predict_proba(input_df)[0][1]
        pred = local_model.predict(input_df)[0]

        if pred == 1:
            status = "🚨 AT-RISK (Needs Academic Support / Intervention)"
            color_markdown = f"<h3 style='color:red;'>{status}</h3>"
        else:
            status = "✅ SAFE (Student is on a Passing Track)"
            color_markdown = f"<h3 style='color:green;'>{status}</h3>"

        result_text = f"{color_markdown}<h4>Calculated Failure Probability: <b>{prob * 100:.1f}%</b></h4>"
        return result_text
    except Exception as e:
        return f"<h3>Error during prediction: {str(e)}</h3>"

# 3. Build Gradio UI
app = gr.Interface(
    fn=test_student_prediction,
    inputs=[
        gr.Slider(1, 4, step=1, value=2, label="Weekly Study Time (1: <2hrs, 2: 2-5hrs, 3: 5-10hrs, 4: >10hrs)"),
        gr.Slider(0, 3, step=1, value=0, label="Past Class Failures"),
        gr.Slider(0, 30, step=1, value=4, label="Number of School Absences"),
        gr.Slider(1, 5, step=1, value=3, label="Current Health Status (1 to 5)"),
        gr.Radio(["yes", "no"], value="yes", label="Internet Access at Home"),
        gr.Radio(["yes", "no"], value="yes", label="Plans for Higher Education"),
        gr.Radio(["yes", "no"], value="no", label="In a Romantic Relationship")
    ],
    outputs=gr.HTML(label="Live Prediction Result"),
    title="🎓 Student Performance Live Risk Predictor",
    description="Adjust the sliders and options below, then click Submit to see the instant model prediction and risk score!"
)

# 4. Launch with shareable link
app.launch(inline=False, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d5cb9807451666cbbd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
